# RAG - Assignment :    

In [220]:
!pip install pypdf sentence-transformers google-genai

In [221]:
# from kaggle_secrets import UserSecretsClient

# GEMINI_API_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY")

from kaggle_secrets import UserSecretsClient

GEMINI_API_KEY = UserSecretsClient().get_secret(
    "gemini_api_key"  # Use the exact name shown in Kaggle Secrets
)

assert GEMINI_API_KEY, "Gemini API key was not loaded."
print("Gemini API key loaded successfully.")

Gemini API key loaded successfully.


## 1. PDF / Dataset : 

In [222]:
from pathlib import Path
from pypdf import PdfReader

PDF_PATH = Path("/kaggle/input/datasets/ksaiveekshith3306/lma-rag-assignment-data/Null_Signal_Games_Netrunner_Comprehensive_Rules_v26.03.pdf")

# basically verifying the path of the pdf
if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF not found: {PDF_PATH}")


reader = PdfReader(str(PDF_PATH))

print("Number of pages:", len(reader.pages))

Number of pages: 99


In [223]:
pages_text = []

for page_num, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    pages_text.append(text)

    if page_num % 100 == 0:
        print(f"Processed {page_num} pages")

print("Total pages:", len(pages_text))
print("Pages with extracted text:", sum(bool(t.strip()) for t in pages_text))
print("Total characters:", sum(len(t) for t in pages_text))

Total pages: 99
Pages with extracted text: 99
Total characters: 370419


In [224]:
# Preview text from the first three pages
for i in range(min(3, len(pages_text))):
    print(f"\n{'=' * 20} PAGE {i + 1} {'=' * 20}\n")
    print(pages_text[i][:1500])


==================== PAGE 1 ====================

Netrunner Comprehensive Rules
Null Signal Games
This rules document is to be used as reference material. It is not intended to be read straight through. If you
still have questions after consulting this document, please ask us online via email.
This version of the Comprehensive Rules document is effective 02 March 2026 .
Summary of Changes
To see this document with a complete list of changes and with new material highlighted, please visit the
Comprehensive Rules Hub at 〈nullsignal.games/comp-rules〉.
• Added rule 1.5.2b to support Méliès U’s ability.
• Clarify link between rules 1.8.4b and 8.1.4 referencing facedown installed Runner cards.
• Added rule 1.13.3 and rule 1.13.12 to clarify properties of hosted objects.
• Added rule 1.21.3a to clarify that revealing does not turn cards faceup.
• Updated the list of subtypes (section 2.16.7) to remove illicit and add liability.
• Added section 2.18 to describe the server indicators on the ba

## Chunking (rule-aware)

The rules PDF is structured around numbered rules, sub-rules, and examples. Fixed character slicing can split a rule or separate it from its exception/example. The chunker below prefers rule/paragraph boundaries, keeps page provenance, and uses a target size with a hard maximum. Both retrieval pipelines use this exact same corpus.


In [225]:
import re

# Tune these after inspecting the chunk previews.
TARGET_CHUNK_CHARS = 1400
MAX_CHUNK_CHARS = 1800
MIN_CHUNK_CHARS = 250

# A rule heading such as "6.5.7. Fully Break" or "9.8.7."
RULE_START_RE = re.compile(
    r"(?m)^\s*\d+(?:\.\d+){1,3}[a-z]?(?:\.)?\s+"
)

def split_page_into_units(page_text, page_num):
    """Split extracted PDF text into rule/paragraph-sized units."""
    text = re.sub(r"[ \t]+", " ", page_text or "").strip()
    if not text:
        return []

    # Prefer numbered rule boundaries. If a page has no rule headings,
    # fall back to blank-line paragraph boundaries.
    starts = [m.start() for m in RULE_START_RE.finditer(text)]
    if len(starts) >= 2:
        boundaries = sorted(set([0] + starts + [len(text)]))
        pieces = [
            text[boundaries[i]:boundaries[i + 1]].strip()
            for i in range(len(boundaries) - 1)
        ]
    else:
        pieces = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]

    return [
        {"text": piece, "pages": [page_num]}
        for piece in pieces if piece
    ]

def split_oversized_unit(unit, max_chars=MAX_CHUNK_CHARS):
    """Split a very long rule at paragraph/sentence/space boundaries."""
    text = unit["text"]
    if len(text) <= max_chars:
        return [unit]

    parts = []
    remaining = text
    while len(remaining) > max_chars:
        cut = remaining.rfind("\n", 0, max_chars)
        if cut < max_chars * 0.5:
            cut = remaining.rfind(". ", 0, max_chars)
        if cut < max_chars * 0.5:
            cut = remaining.rfind(" ", 0, max_chars)
        if cut < max_chars * 0.5:
            cut = max_chars

        piece = remaining[:cut].strip()
        if piece:
            parts.append({"text": piece, "pages": unit["pages"][:]})
        remaining = remaining[cut:].strip()

    if remaining:
        parts.append({"text": remaining, "pages": unit["pages"][:]})
    return parts

# Keep the original page text as the source of truth and preserve page metadata.
units = []
for page_num, page_text in enumerate(pages_text, start=1):
    units.extend(split_page_into_units(page_text, page_num))

# Avoid oversized chunks, then pack adjacent units to a target size.
bounded_units = []
for unit in units:
    bounded_units.extend(split_oversized_unit(unit))

chunks = []
current_texts = []
current_pages = []
current_len = 0

def flush_chunk():
    global current_texts, current_pages, current_len
    if current_texts:
        text = "\n\n".join(current_texts).strip()
        if text:
            chunks.append({
                "text": text,
                "page": min(current_pages),
                "pages": sorted(set(current_pages)),
                "start_char": None,
                "end_char": None
            })
    current_texts = []
    current_pages = []
    current_len = 0

for unit in bounded_units:
    unit_text = unit["text"]
    extra_len = len(unit_text) + (2 if current_texts else 0)

    if current_texts and current_len + extra_len > TARGET_CHUNK_CHARS:
        flush_chunk()

    # A single unit may be larger than target but has already been capped.
    current_texts.append(unit_text)
    current_pages.extend(unit["pages"])
    current_len += extra_len

flush_chunk()

# Small final chunks are merged into the previous chunk when that remains safe.
if len(chunks) > 1 and len(chunks[-1]["text"]) < MIN_CHUNK_CHARS:
    last = chunks[-1]
    previous = chunks[-2]
    if len(previous["text"]) + 2 + len(last["text"]) <= MAX_CHUNK_CHARS:
        previous["text"] += "\n\n" + last["text"]
        previous["pages"] = sorted(set(previous["pages"] + last["pages"]))
        previous["page"] = min(previous["pages"])
        chunks.pop()

print("Pages with text:", sum(bool(t.strip()) for t in pages_text))
print("Total chunks:", len(chunks))
print("Average chunk characters:", round(sum(len(c["text"]) for c in chunks) / max(len(chunks), 1)))
print("Min / max chunk characters:", min(map(lambda c: len(c["text"]), chunks)), "/", max(map(lambda c: len(c["text"]), chunks)))
print("First chunk pages:", chunks[0]["pages"] if chunks else [])
print((chunks[0]["text"][:1200] if chunks else "No chunks generated."))


Pages with text: 99
Total chunks: 315
Average chunk characters: 1177
Min / max chunk characters: 106 / 1800
First chunk pages: [1]
Netrunner Comprehensive Rules
Null Signal Games
This rules document is to be used as reference material. It is not intended to be read straight through. If you
still have questions after consulting this document, please ask us online via email.
This version of the Comprehensive Rules document is effective 02 March 2026 .
Summary of Changes
To see this document with a complete list of changes and with new material highlighted, please visit the
Comprehensive Rules Hub at 〈nullsignal.games/comp-rules〉.
• Added rule 1.5.2b to support Méliès U’s ability.
• Clarify link between rules 1.8.4b and 8.1.4 referencing facedown installed Runner cards.
• Added rule 1.13.3 and rule 1.13.12 to clarify properties of hosted objects.
• Added rule 1.21.3a to clarify that revealing does not turn cards faceup.
• Updated the list of subtypes (section 2.16.7) to remove illicit and

In [226]:
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i + 1} ---")
    print("Page:", chunk["page"])
    print("Characters:", len(chunk["text"]))
    print(chunk["text"][:300])


--- Chunk 1 ---
Page: 1
Characters: 1758
Netrunner Comprehensive Rules
Null Signal Games
This rules document is to be used as reference material. It is not intended to be read straight through. If you
still have questions after consulting this document, please ask us online via email.
This version of the Comprehensive Rules document is eff

--- Chunk 2 ---
Page: 1
Characters: 1326
• Clarified rules about prohibitions on accesses in section 7.4.2 and rule 7.4.6a.
• Expanded section 8.3.3 to add a rule ensuring that cards about to be arranged in any order don’t need
to have their order maintained for other effects in the same instruction.
• Updated rule 8.5.12 to account for “C

--- Chunk 3 ---
Page: 2
Characters: 1360
Table of Contents
Summary of Changes . . . . . . . . . . . . . . 1
Table of Contents . . . . . . . . . . . . . . . . 2
1 Game Concepts . . . . . . . . . . . . . . . 4

1.1 General . . . . . . . . . . . . . . . . . 4

1.2 Golden Rules . . . . . . . . . . . . . . 4

1.3 Sym

In [227]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = model.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Number of chunks:", len(chunk_texts))
print("Embeddings shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Number of chunks: 315
Embeddings shape: (315, 384)


In [228]:
import numpy as np
from sentence_transformers import util

query = "What happens to subroutines gained during an encounter with ice?"

query_embedding = model.encode(
    query,
    convert_to_numpy=True
)

similarities = util.cos_sim(
    query_embedding,
    embeddings
)[0].cpu().numpy()

top_k = 3
top_indices = np.argsort(similarities)[::-1][:top_k]

for rank, idx in enumerate(top_indices, start=1):
    print(f"\n--- Result {rank} ---")
    print("Page:", chunks[idx]["page"])
    print("Similarity:", round(float(similarities[idx]), 4))
    print(chunks[idx]["text"][:700])


--- Result 1 ---
Page: 76
Similarity: 0.7272
subroutines are added after other subroutines or does not specify an order. Subroutines in
this category on a single piece of ice are always identical. If the number of such subroutines
increases, the new copies are added after any previously-existing subroutines in this category.
If the number of such subroutines decreases, subroutines are lost beginning with the last
subroutine the ice previously had from the same ability and working backwards.
Example: The Corp has 3 cards in HQ while the Runner is encountering Ashigaru. After the
Runner breaks Ashigaru’s 3 subroutines, the Corp uses Panic Button to draw another card. The
new subroutine is added after the previous 3, so the Runner can use

--- Result 2 ---
Page: 46
Similarity: 0.6936
6.9.3c never occurring, meaning that subroutines on the encountered ice are neither broken
nor resolved.
c. A few bypass effects can be triggered later. If ice is bypassed in step 6.9.3b, subroutines
may or 

## Baseline:

In [229]:
import numpy as np
from sentence_transformers import util

def baseline_retrieve(query, top_k=10):
    query_embedding = model.encode(
        query,
        convert_to_numpy=True
    )

    scores = util.cos_sim(
        query_embedding,
        embeddings
    )[0].cpu().numpy()

    indices = np.argsort(scores)[::-1][:top_k]

    return [
        {
            "chunk_id": int(idx),
            "page": chunks[idx]["page"],
            "pages": chunks[idx].get("pages", [chunks[idx]["page"]]),
            "text": chunks[idx]["text"],
            "score": float(scores[idx])
        }
        for idx in indices
    ]


query = "What happens to subroutines gained during an encounter with ice?"

candidates = baseline_retrieve(query, top_k=10)

print("Top 10 retrieved candidates:\n")

for rank, result in enumerate(candidates, start=1):
    print(
        f"Rank {rank} | "
        f"Chunk {result['chunk_id']} | "
        f"Page {result['page']} | "
        f"Score {result['score']:.4f}"
    )

print("\n--- Baseline Top 3 Context ---")

baseline_top3 = candidates[:3]

for result in baseline_top3:
    print(f"\nPage {result['page']}\n{result['text']}")

Top 10 retrieved candidates:

Rank 1 | Chunk 244 | Page 76 | Score 0.7272
Rank 2 | Chunk 139 | Page 46 | Score 0.6936
Rank 3 | Chunk 246 | Page 77 | Score 0.6899
Rank 4 | Chunk 138 | Page 45 | Score 0.6846
Rank 5 | Chunk 149 | Page 49 | Score 0.6581
Rank 6 | Chunk 240 | Page 75 | Score 0.6385
Rank 7 | Chunk 312 | Page 97 | Score 0.6217
Rank 8 | Chunk 137 | Page 45 | Score 0.6053
Rank 9 | Chunk 241 | Page 76 | Score 0.5887
Rank 10 | Chunk 242 | Page 76 | Score 0.5845

--- Baseline Top 3 Context ---

Page 76
subroutines are added after other subroutines or does not specify an order. Subroutines in
this category on a single piece of ice are always identical. If the number of such subroutines
increases, the new copies are added after any previously-existing subroutines in this category.
If the number of such subroutines decreases, subroutines are lost beginning with the last
subroutine the ice previously had from the same ability and working backwards.
Example: The Corp has 3 cards in HQ w

## Advanced using Re-ranking:

In [230]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

def rerank_candidates(query, candidates, top_k=3):
    pairs = [
        (query, candidate["text"])
        for candidate in candidates
    ]

    scores = reranker.predict(pairs)

    reranked = []

    for candidate, score in zip(candidates, scores):
        item = candidate.copy()
        item["rerank_score"] = float(score)
        reranked.append(item)

    reranked.sort(
        key=lambda item: item["rerank_score"],
        reverse=True
    )

    return reranked[:top_k]


reranked_top3 = rerank_candidates(
    query,
    candidates,
    top_k=3
)

print("Cross-Encoder Top 3:\n")

for rank, result in enumerate(reranked_top3, start=1):
    print(
        f"Rank {rank} | "
        f"Chunk {result['chunk_id']} | "
        f"Page {result['page']} | "
        f"Score {result['rerank_score']:.4f}"
    )

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Cross-Encoder Top 3:

Rank 1 | Chunk 246 | Page 77 | Score 7.3383
Rank 2 | Chunk 138 | Page 45 | Score 5.7131
Rank 3 | Chunk 149 | Page 49 | Score 5.3986


In [231]:
# import pandas as pd

# baseline_ids = [c["chunk_id"] for c in candidates]
# reranked_ids = [c["chunk_id"] for c in reranked_top3]

# comparison = pd.DataFrame({
#     "Chunk ID": baseline_ids,
#     "Page": [c["page"] for c in candidates],
#     "Baseline Rank": range(1, len(candidates) + 1),
#     "Cosine Score": [c["score"] for c in candidates],
#     "Cross-Encoder Rank": [
#         next(
#             (rank for rank, item in enumerate(
#                 sorted(
#                     [
#                         {
#                             **c,
#                             "rerank_score": float(s)
#                         }
#                         for c, s in zip(
#                             candidates,
#                             reranker.predict([
#                                 (query, c["text"]) for c in candidates
#                             ])
#                         )
#                     ],
#                     key=lambda x: x["rerank_score"],
#                     reverse=True
#                 ),
#                 start=1
#             ) if item["chunk_id"] == c["chunk_id"]),
#             None
#         )
#         for c in candidates
#     ]
# })

# print(comparison.to_string(index=False))

import pandas as pd

cross_scores = reranker.predict([
    (query, candidate["text"])
    for candidate in candidates
])

reranked_all = sorted(
    [
        {**candidate, "rerank_score": float(score)}
        for candidate, score in zip(candidates, cross_scores)
    ],
    key=lambda item: item["rerank_score"],
    reverse=True
)

cross_rank = {
    item["chunk_id"]: rank
    for rank, item in enumerate(reranked_all, start=1)
}

comparison = pd.DataFrame([
    {
        "Chunk ID": item["chunk_id"],
        "PDF Page": item["page"],
        "Baseline Rank": rank,
        "Cross-Encoder Rank": cross_rank[item["chunk_id"]],
        "Baseline Score": round(item["score"], 4),
        "Cross-Encoder Score": round(
            next(
                x["rerank_score"]
                for x in reranked_all
                if x["chunk_id"] == item["chunk_id"]
            ), 4
        )
    }
    for rank, item in enumerate(candidates, start=1)
])

print(comparison.to_string(index=False))

# Keep the Top 3 from the complete reranked list
reranked_top3 = reranked_all[:3]

 Chunk ID  PDF Page  Baseline Rank  Cross-Encoder Rank  Baseline Score  Cross-Encoder Score
      244        76              1                   7          0.7272               4.3738
      139        46              2                   5          0.6936               4.8798
      246        77              3                   1          0.6899               7.3383
      138        45              4                   2          0.6846               5.7131
      149        49              5                   3          0.6581               5.3986
      240        75              6                   9          0.6385              -1.1277
      312        97              7                  10          0.6217              -5.0897
      137        45              8                   4          0.6053               5.1560
      241        76              9                   8          0.5887               3.3387
      242        76             10                   6          0.5845          

In [232]:
LLM_BACKEND = "gemini"
GEMINI_MODEL = "gemini-3.5-flash-lite"

In [233]:
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)

response = client.models.generate_content(
    model=GEMINI_MODEL,
    contents="Reply with exactly: Gemini connection successful."
)

print(response.text)

Gemini connection successful.


In [234]:
def generate_answer(prompt):
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt
    )
    return response.text


def build_prompt(query, retrieved_chunks):
    context = "\n\n".join(
        f"[Chunk {item['chunk_id']} | PDF page {item['page']}]\n"
        f"{item['text']}"
        for item in retrieved_chunks
    )

    return f"""Answer the question using only the supplied context.

If the context does not contain enough information, state that
the answer cannot be determined from the provided context.
Do not invent rules or facts.

Question:
{query}

Context:
{context}

Answer:"""

In [235]:
import time
import pandas as pd

query = "What happens to subroutines gained during an encounter with ice?"

# Measure complete pipeline latency consistently, including LLM generation.
start = time.perf_counter()
baseline_candidates = baseline_retrieve(query, top_k=10)
baseline_retrieval_time = time.perf_counter() - start
baseline_top3 = baseline_candidates[:3]
baseline_prompt = build_prompt(query, baseline_top3)

start = time.perf_counter()
answer1 = generate_answer(baseline_prompt)
baseline_llm_time = time.perf_counter() - start
baseline_total_time = baseline_retrieval_time + baseline_llm_time

start = time.perf_counter()
advanced_candidates = baseline_retrieve(query, top_k=10)
advanced_retrieval_time = time.perf_counter() - start

start = time.perf_counter()
advanced_reranked = rerank_candidates(query, advanced_candidates, top_k=3)
advanced_reranking_time = time.perf_counter() - start
reranked_top3 = advanced_reranked
reranked_prompt = build_prompt(query, reranked_top3)

start = time.perf_counter()
answer2 = generate_answer(reranked_prompt)
reranked_llm_time = time.perf_counter() - start
reranked_total_time = advanced_retrieval_time + advanced_reranking_time + reranked_llm_time

print("===== ANSWER 1: BASELINE TOP 3 =====\n")
print(answer1)
print("\n===== ANSWER 2: CROSS-ENCODER RERANKED TOP 3 =====\n")
print(answer2)

latency_table = pd.DataFrame([
    {
        "Pipeline": "Baseline",
        "Bi-encoder retrieval (s)": baseline_retrieval_time,
        "Cross-encoder reranking (s)": 0.0,
        "LLM generation (s)": baseline_llm_time,
        "Total end-to-end (s)": baseline_total_time
    },
    {
        "Pipeline": "Baseline + Cross-Encoder",
        "Bi-encoder retrieval (s)": advanced_retrieval_time,
        "Cross-encoder reranking (s)": advanced_reranking_time,
        "LLM generation (s)": reranked_llm_time,
        "Total end-to-end (s)": reranked_total_time
    }
])
display(latency_table.round(4))


===== ANSWER 1: BASELINE TOP 3 =====

Based on the provided context, when a piece of ice gains 1 or more new subroutines during an encounter with it, those subroutines come into existence unbroken with respect to that encounter. Additionally, if the new subroutine is of a certain category where the number of such subroutines increases, the new copies are added after any previously-existing subroutines in that category.

===== ANSWER 2: CROSS-ENCODER RERANKED TOP 3 =====

Based on the provided context, when a piece of ice gains 1 or more new subroutines during an encounter with it, those subroutines come into existence unbroken with respect to that encounter.


,Pipeline,Bi-encoder retrieval (s),Cross-encoder reranking (s),LLM generation (s),Total end-to-end (s)
0,Baseline,0.0184,0.0000,4.5195,4.5379
1,Baseline + Cross-Encoder,0.0152,1.0064,0.7359,1.7575


## Section 3: Failure-mode analysis

The queries below target document-specific rules, exceptions, and constraints. For each query, the notebook retrieves the same Top 10 candidate set for both approaches, records the baseline rank and cross-encoder rank for every candidate, and shows the Top 3 supplied to the LLM.

**Important:** a query only counts as a successful failure-mode example if the baseline Top 3 misses or inadequately represents the critical rule and re-ranking promotes the relevant chunk into its Top 3. Verify the displayed text and answers; do not claim a rank improvement unless the actual run demonstrates it.


In [236]:
import time
import pandas as pd

failure_queries = [
    {
        "type": "Added subroutines / corner case",
        "query": "What happens to subroutines gained during an encounter with ice?"
    },
    {
        "type": "Negation / constraint",
        "query": "Can a subroutine that has already been broken be broken again?"
    },
    {
        "type": "Exception / bypass",
        "query": "When the Runner bypasses a piece of ice, what happens to the encounter and its subroutines?"
    },
    {
        "type": "Targeting already-broken subroutines",
        "query": "Can an ability that breaks subroutines target a subroutine that is already broken?"
    },
    {
        "type": "Fully-broken ice / exception",
        "query": "If a piece of ice gains new subroutines after the Runner has fully broken it, does the Runner lose the fully-broken status, and can breaking those new subroutines fully break it again?"
    },
    {
        "type": "Keyword Trap / Distractor",
        "query": (
            "If the Runner breaks all subroutines on a piece of ice "
            "using abilities from different cards, does any single "
            "card count as having fully broken the ice?"
        )
    },
    {
        "type": "Exceptions & Corner Cases",
        "query": (
            "If a piece of ice gains new subroutines after the Runner "
            "has fully broken it, does breaking those new subroutines "
            "fully break the ice again?"
        )
    },
    {
        "type": "Negation & Constraints",
        "query": (
            "Can an ability that breaks all subroutines be used if "
            "every subroutine on the encountered ice is already broken?"
        )
    },
    {
        "type": "Other Document-Specific Failures",
        "query": (
            "If the encountered ice has no subroutines, when is the "
            "Runner considered to have fully broken it, and does any "
            "card or other object fully break it?"
        )
    },
    {
        "type": "Other Document-Specific Failure",
        "query": "If the Runner bypasses ice after the paid ability window has begun, do any remaining unbroken subroutines resolve?"
    },
    {
        "type": "Exceptions & Corner Cases",
        "query": (
            "If multiple abilities trigger at the same time during a turn, "
            "how is the order of resolving those abilities determined?"
        )
    },
    {
        "type": "Negation & Constraints",
        "query": (
            "What happens if a player attempts an action that violates a "
            "restriction in the game rules, and can that action still resolve?"
        )
    },
    {
        "type": "Keyword Trap / Distractor",
        "query": (
            "If a card effect makes another card unable to be used, does "
            "that mean the affected card is trashed, or does it remain in play?"
        )
    },
    {
        "type": "Out-of-Domain",
        "query": "What is the best opening strategy for winning a game of chess?"
    }
]

failure_results = []
failure_rank_tables = {}

for case in failure_queries:
    test_query = case["query"]
    print("\n" + "=" * 100)
    print(f"TYPE: {case['type']}\nQUERY: {test_query}")

    start = time.perf_counter()
    top10 = baseline_retrieve(test_query, top_k=10)
    retrieval_time = time.perf_counter() - start

    start = time.perf_counter()
    scores = reranker.predict([(test_query, item["text"]) for item in top10])
    reranking_time = time.perf_counter() - start

    reranked_all = sorted(
        [
            {**item, "rerank_score": float(score)}
            for item, score in zip(top10, scores)
        ],
        key=lambda item: item["rerank_score"],
        reverse=True
    )

    cross_ranks = {
        item["chunk_id"]: rank
        for rank, item in enumerate(reranked_all, start=1)
    }

    cross_scores_by_id = {
        item["chunk_id"]: item["rerank_score"]
        for item in reranked_all
    }

    rank_table = pd.DataFrame([
        {
            "Chunk": item["chunk_id"],
            "PDF page(s)": item.get("pages", [item["page"]]),
            "Baseline rank": rank,
            "Cross-encoder rank": cross_ranks[item["chunk_id"]],
            "Bi-encoder score": round(item["score"], 4),
            "Cross-encoder score": round( cross_scores_by_id[item["chunk_id"]], 4)
        }
        for rank, item in enumerate(top10, start=1)
    ])
    failure_rank_tables[test_query] = rank_table

    baseline_top3 = top10[:3]
    reranked_top3 = reranked_all[:3]

    print("\nRank changes (Cross-encoder rank is within the original Top 10):")
    display(rank_table)
    print("Baseline Top 3 chunk IDs:", [x["chunk_id"] for x in baseline_top3])
    print("Re-ranked Top 3 chunk IDs:", [x["chunk_id"] for x in reranked_top3])
    print(f"Bi-encoder retrieval time: {retrieval_time:.4f} s")
    print(f"Cross-encoder reranking time: {reranking_time:.4f} s")

    print("\n--- Baseline Top 3 passages ---")
    for rank, item in enumerate(baseline_top3, start=1):
        print(f"\nRank {rank} | Chunk {item['chunk_id']} | Page(s) {item.get('pages', [item['page']])}")
        print(item["text"])

    print("\n--- Re-ranked Top 3 passages ---")
    for rank, item in enumerate(reranked_top3, start=1):
        print(f"\nRank {rank} | Chunk {item['chunk_id']} | Page(s) {item.get('pages', [item['page']])}")
        print(item["text"])

    failure_results.append({
        "Failure type": case["type"],
        "Query": test_query,
        "Baseline Top 3": [x["chunk_id"] for x in baseline_top3],
        "Re-ranked Top 3": [x["chunk_id"] for x in reranked_top3],
        "Retrieval time (s)": retrieval_time,
        "Reranking time (s)": reranking_time
    })

failure_summary = pd.DataFrame(failure_results)
display(failure_summary)



TYPE: Added subroutines / corner case
QUERY: What happens to subroutines gained during an encounter with ice?

Rank changes (Cross-encoder rank is within the original Top 10):


,Chunk,PDF page(s),Baseline rank,Cross-encoder rank,Bi-encoder score,Cross-encoder score
0,244,[76],1,7,0.7272,4.3738
1,139,[46],2,5,0.6936,4.8798
2,246,[77],3,1,0.6899,7.3383
3,138,"[45, 46]",4,2,0.6846,5.7131
4,149,[49],5,3,0.6581,5.3986
5,240,"[75, 76]",6,9,0.6385,-1.1277
6,312,[97],7,10,0.6217,-5.0897
7,137,[45],8,4,0.6053,5.1560
8,241,[76],9,8,0.5887,3.3387
9,242,[76],10,6,0.5845,4.7895


Baseline Top 3 chunk IDs: [244, 139, 246]
Re-ranked Top 3 chunk IDs: [246, 138, 149]
Bi-encoder retrieval time: 0.0184 s
Cross-encoder reranking time: 0.9974 s

--- Baseline Top 3 passages ---

Rank 1 | Chunk 244 | Page(s) [76]
subroutines are added after other subroutines or does not specify an order. Subroutines in
this category on a single piece of ice are always identical. If the number of such subroutines
increases, the new copies are added after any previously-existing subroutines in this category.
If the number of such subroutines decreases, subroutines are lost beginning with the last
subroutine the ice previously had from the same ability and working backwards.
Example: The Corp has 3 cards in HQ while the Runner is encountering Ashigaru. After the
Runner breaks Ashigaru’s 3 subroutines, the Corp uses Panic Button to draw another card. The
new subroutine is added after the previous 3, so the Runner can use Gbahali to break it.
76

Rank 2 | Chunk 139 | Page(s) [46]
6.9.3c never

,Chunk,PDF page(s),Baseline rank,Cross-encoder rank,Bi-encoder score,Cross-encoder score
0,246,[77],1,4,0.6632,5.1523
1,247,[77],2,2,0.6185,6.8427
2,139,[46],3,1,0.5685,7.1767
3,138,"[45, 46]",4,6,0.5201,3.9920
4,244,[76],5,5,0.4858,4.1145
5,131,[44],6,7,0.4827,2.4237
6,249,[78],7,9,0.4577,0.4668
7,240,"[75, 76]",8,10,0.4555,-6.6797
8,248,"[77, 78]",9,8,0.4306,2.1804
9,245,[77],10,3,0.4193,5.4332


Baseline Top 3 chunk IDs: [246, 247, 139]
Re-ranked Top 3 chunk IDs: [139, 247, 245]
Bi-encoder retrieval time: 0.0154 s
Cross-encoder reranking time: 0.7643 s

--- Baseline Top 3 passages ---

Rank 1 | Chunk 246 | Page(s) [77]
9.8.4. For each encounter, the subroutines on the encountered ice have a status ofBROKEN or UNBROKEN
with respect to that encounter.
a. When an encounter begins, each of the encountered ice’s subroutines becomes unbroken
with respect to that encounter.
b. When a piece of ice gains 1 or more new subroutines during an encounter with it, those
subroutines come into existence unbroken with respect to that encounter.
c. When an encounter ends, card abilities can still check whether subroutines were broken or
unbroken during that encounter.

9.8.5. To BREAK a subroutine is to change its status with respect to the active encounter from unbroken to
broken. Many cards that can allow a player to break subroutines are icebreakers (seesection 3.9.5).
If the Runner breaks al

...

,Chunk,PDF page(s),Baseline rank,Cross-encoder rank,Bi-encoder score,Cross-encoder score
0,187,[60],1,1,0.7309,5.5361
1,186,[60],2,2,0.5649,3.7070
2,170,[55],3,6,0.5595,1.5209
3,152,[50],4,9,0.5504,-1.1680
4,283,[88],5,3,0.5493,2.6561
5,281,[87],6,5,0.5421,1.9769
6,294,[91],7,4,0.5253,2.0536
7,107,"[36, 37]",8,10,0.5247,-4.0003
8,32,[11],9,7,0.5138,1.1337
9,116,[39],10,8,0.5008,0.2434


Baseline Top 3 chunk IDs: [187, 186, 170]
Re-ranked Top 3 chunk IDs: [187, 186, 283]
Bi-encoder retrieval time: 0.0185 s
Cross-encoder reranking time: 0.8413 s

--- Baseline Top 3 passages ---

Rank 1 | Chunk 187 | Page(s) [60]
8.6.6. In certain circumstances, a played card is not trashed after resolving it.
a. If the played card is no longer in the play area when step 8.6.7g is reached, it is not trashed.
Example: The Runner plays Ashen Epilogue and resolves its play abilities. Its last ability removes
it from the game.
b. If the played card has been converted to a condition counter or otherwise hosted on a card
while resolving its play abilities, it is not trashed. See also rule 10.1.4.
c. If the played card has an ability indicating that it is not trashed until a certain effect occurs,
instead of trashing it, a lingering effect is created with a duration that expires as the indicated
effect occurs. See also section 9.10.
d. If an ability that plays an event or operation also contain

,Chunk,PDF page(s),Baseline rank,Cross-encoder rank,Bi-encoder score,Cross-encoder score
0,5,"[2, 3]",1,5,0.2974,-10.6435
1,313,[98],2,10,0.2823,-11.2027
2,18,[7],3,2,0.2753,-5.2868
3,148,"[48, 49]",4,7,0.2505,-11.1273
4,141,"[46, 47]",5,6,0.2501,-11.1103
5,19,[7],6,1,0.2488,-5.0245
6,4,[2],7,3,0.2473,-9.5131
7,129,[43],8,8,0.2452,-11.1336
8,144,[47],9,4,0.2390,-10.2805
9,124,[42],10,9,0.2353,-11.1670


Baseline Top 3 chunk IDs: [5, 313, 18]
Re-ranked Top 3 chunk IDs: [19, 18, 4]
Bi-encoder retrieval time: 0.0150 s
Cross-encoder reranking time: 1.2196 s

--- Baseline Top 3 passages ---

Rank 1 | Chunk 5 | Page(s) [2, 3]
6.1 General . . . . . . . . . . . . . . . . . 41

6.2 Position . . . . . . . . . . . . . . . . 42

6.3 Initiation . . . . . . . . . . . . . . . . 44

6.4 Approach Ice . . . . . . . . . . . . . 45

6.5 Encounter Ice . . . . . . . . . . . . . 45

6.5.7 Fully Break . . . . . . . . . . 46

6.5.8 Bypass . . . . . . . . . . . . 46

6.5.9 Forced Encounters . . . . . 46

6.6 Movement . . . . . . . . . . . . . . . 46

6.7 Success . . . . . . . . . . . . . . . . 47

6.8 Run Ends Phase . . . . . . . . . . . 47

6.9 Steps of a Run . . . . . . . . . . . . 48
7 Accessing Cards and Breaching Servers 49

7.1 Accessing Cards . . . . . . . . . . . . 50

7.2 Steps of Accessing a Card . . . . . 50

7.3 Breaching Servers . . . . . . . . . . 51

7.4 Determining Candidates . . . . . . 52

7.

,Failure type,Query,Baseline Top 3,Re-ranked Top 3,Retrieval time (s),Reranking time (s)
0,Added subroutines / corner case,What happens to subroutines gained during an e...,"[244, 139, 246]","[246, 138, 149]",0.018368,0.997394
1,Negation / constraint,Can a subroutine that has already been broken ...,"[246, 247, 139]","[139, 247, 245]",0.015432,0.764322
2,Exception / bypass,"When the Runner bypasses a piece of ice, what ...","[138, 149, 133]","[138, 139, 246]",0.017456,0.941293
3,Targeting already-broken subroutines,Can an ability that breaks subroutines target ...,"[247, 246, 139]","[247, 245, 246]",0.016271,0.768970
4,Fully-broken ice / exception,If a piece of ice gains new subroutines after ...,"[246, 138, 244]","[138, 244, 131]",0.021665,0.822849
5,Keyword Trap / Distractor,If the Runner breaks all subroutines on a piec...,"[138, 246, 247]","[247, 244, 138]",0.019064,0.820072
6,Exceptions & Corner Cases,If a piece of ice gains new subroutines after ...,"[138, 246, 244]","[138, 244, 246]",0.020978,0.821904
7,Negation & Constraints,Can an ability that breaks all subroutines be ...,"[246, 247, 138]","[247, 246, 139]",0.019228,1.063670
8,Other Document-Specific Failures,"If the encountered ice has no subroutines, whe...","[138, 246, 139]","[138, 139, 246]",0.019614,0.787858
9,Other Document-Specific Failure,If the Runner bypasses ice after the paid abil...,"[149, 138, 312]","[138, 149, 137]",0.018735,0.803434


### Compare LLM answers for the failure queries
Run this after the rank-analysis cell. It calls the LLM twice per query, using the identical prompt template and each pipeline's own Top 3. Inspect whether the baseline misses a rule, while the re-ranked answer correctly includes it.


In [237]:
# This cell makes 2 LLM calls per query. Set the list to only the confirmed
# failure cases if you want to reduce API usage.
failure_answer_comparisons = []

for case in failure_queries:
    time.sleep(10)
    test_query = case["query"]
    top10 = baseline_retrieve(test_query, top_k=10)
    reranked_top3 = rerank_candidates(test_query, top10, top_k=3)
    baseline_top3 = top10[:3]

    start = time.perf_counter()
    answer_baseline = generate_answer(build_prompt(test_query, baseline_top3))
    baseline_answer_time = time.perf_counter() - start

    start = time.perf_counter()
    answer_reranked = generate_answer(build_prompt(test_query, reranked_top3))
    reranked_answer_time = time.perf_counter() - start

    print("\n" + "=" * 100)
    print("FAILURE TYPE:", case["type"])
    print("QUERY:", test_query)
    print("\n--- BASELINE ANSWER ---\n")
    print(answer_baseline)
    print("\n--- RE-RANKED ANSWER ---\n")
    print(answer_reranked)
    print(f"\nLLM latency — baseline: {baseline_answer_time:.3f} s; re-ranked: {reranked_answer_time:.3f} s")

    failure_answer_comparisons.append({
        "Failure type": case["type"],
        "Query": test_query,
        "Baseline Top 3 IDs": [x["chunk_id"] for x in baseline_top3],
        "Re-ranked Top 3 IDs": [x["chunk_id"] for x in reranked_top3],
        "Baseline LLM time (s)": baseline_answer_time,
        "Re-ranked LLM time (s)": reranked_answer_time,
        "Baseline answer": answer_baseline,
        "Re-ranked answer": answer_reranked
    })

failure_answers_df = pd.DataFrame(failure_answer_comparisons)
display(failure_answers_df[[
    "Failure type", "Query", "Baseline Top 3 IDs", "Re-ranked Top 3 IDs",
    "Baseline LLM time (s)", "Re-ranked LLM time (s)"
]])



FAILURE TYPE: Added subroutines / corner case
QUERY: What happens to subroutines gained during an encounter with ice?

--- BASELINE ANSWER ---

Based on the provided context, when a piece of ice gains 1 or more new subroutines during an encounter with it, those subroutines come into existence unbroken with respect to that encounter. Additionally, if the new subroutine is of a certain category where the number increases, the new copies are added after any previously-existing subroutines in that category (or after previous subroutines, as seen in the Ashigaru example).

--- RE-RANKED ANSWER ---

Based on the provided context, when a piece of ice gains 1 or more new subroutines during an encounter with it, those subroutines come into existence unbroken with respect to that encounter.

LLM latency — baseline: 0.757 s; re-ranked: 0.641 s

FAILURE TYPE: Negation / constraint
QUERY: Can a subroutine that has already been broken be broken again?

--- BASELINE ANSWER ---

Based on the provided

,Failure type,Query,Baseline Top 3 IDs,Re-ranked Top 3 IDs,Baseline LLM time (s),Re-ranked LLM time (s)
0,Added subroutines / corner case,What happens to subroutines gained during an e...,"[244, 139, 246]","[246, 138, 149]",0.757008,0.640543
1,Negation / constraint,Can a subroutine that has already been broken ...,"[246, 247, 139]","[139, 247, 245]",0.818414,0.971683
2,Exception / bypass,"When the Runner bypasses a piece of ice, what ...","[138, 149, 133]","[138, 139, 246]",0.861805,1.937850
3,Targeting already-broken subroutines,Can an ability that breaks subroutines target ...,"[247, 246, 139]","[247, 245, 246]",0.704043,1.209180
4,Fully-broken ice / exception,If a piece of ice gains new subroutines after ...,"[246, 138, 244]","[138, 244, 131]",1.296234,1.479747
5,Keyword Trap / Distractor,If the Runner breaks all subroutines on a piec...,"[138, 246, 247]","[247, 244, 138]",0.866538,1.112191
6,Exceptions & Corner Cases,If a piece of ice gains new subroutines after ...,"[138, 246, 244]","[138, 244, 246]",1.704130,2.432693
7,Negation & Constraints,Can an ability that breaks all subroutines be ...,"[246, 247, 138]","[247, 246, 139]",10.784585,0.533732
8,Other Document-Specific Failures,"If the encountered ice has no subroutines, whe...","[138, 246, 139]","[138, 139, 246]",0.780722,0.630725
9,Other Document-Specific Failure,If the Runner bypasses ice after the paid abil...,"[149, 138, 312]","[138, 149, 137]",0.553247,0.541993
